# Pathadol Neuro-Symbolic Brain Trainer (Colab Pro L4 GPU)

This 1-click notebook trains **Pathadol's Bedwars + PvP tactical Brain** (`OBS_DIM=72 -> LayerNorm+SiLU -> GRU(128) -> 13 Maneuvers / 4 Sliders / 4 Combat Modes`) on your Colab GPU and exports a compact (~450 KB) **`active.pbrain`** file ready for `.minecraft/pathadol/brain/active.pbrain`.

No ONNX runtime (~75 MB) is needed — Pathadol runs native pure-Java inference directly in `< 0.1 ms`.

In [ ]:
!nvidia-smi
!mkdir -p brain_data out
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Step 1 — Load Training Script & Optional Live `.jsonl` Trajectories
Upload any `.jsonl` files recorded in Minecraft via `#brain record start` into `./brain_data/` (or leave empty to warm-start from the built-in tactical teacher + vectorized L4 PPO).

In [ ]:
# Option A: Clone from your Pathadol repo if public, or upload pathadol_brain_train.py
# !git clone https://github.com/fivesaw/Panadol.git && cp Panadol/pathadol/colab/pathadol_brain_train.py .
from google.colab import files
import os
if not os.path.exists('pathadol_brain_train.py'):
    print('Upload pathadol_brain_train.py (and optional brain_data/*.jsonl logs):')
    uploaded = files.upload()
    for fn in uploaded:
        if fn.endswith('.jsonl'):
            os.replace(fn, os.path.join('brain_data', os.path.basename(fn)))

## Step 2 — Run Behavioral Cloning + Vectorized L4 PPO + `.pbrain` Parity Verification

In [ ]:
!python3 pathadol_brain_train.py --data-glob "brain_data/*.jsonl" --out out/active.pbrain --bc-epochs 10 --ppo-updates 120 --num-envs 512

## Step 3 — Download `active.pbrain` into `.minecraft/pathadol/brain/active.pbrain`
Then run **`#brain reload`** followed by **`#brain start`** or **`#bws auto`** in game.

In [ ]:
from google.colab import files
files.download('out/active.pbrain')